# EasyNCO（`final/`）离线 checkpoint 转换说明（以 MTReLD 为例）

本 notebook 用 **本仓库里的真实文件** 与 **本仓库里的真实代码** 来说明：

- 你的 MTReLD 移植中，“模型参数转换/兼容加载”具体做了什么。
- 转换前（作者 checkpoint）与转换后（平台 checkpoint）的**实际格式**分别是什么。
- 为什么要做离线转换（offline conversion），以及它解决的具体问题。

相关文件：
- 在线加载（当前推理路径）：`final/eval.py` 调用 `final/utils/utils.py:load_model()`
- 离线转换脚本：`final/convert_checkpoint_offline.py`
- 作者 checkpoint：
  - `final/pretrained/pretrained/reld_moe_light/epoch-5000.pt`（平台模型名 `mtreld`）
  - `final/pretrained/pretrained/reld_mtl/epoch-5000.pt`（平台模型名 `mtreld_mtl`）
- 推理日志（证明你现在确实在走作者 `model_state_dict` 分支）：`final/logs/tsplib_70/lehd/my.txt`


## 0. 主线结论（本 notebook 只讲离线转换）

离线转换的目标与结果在平台代码里是硬编码的：

- 平台评测入口 `final/eval.py` 会从 `settings.test_loader` 得到 checkpoint 文件路径，然后加载权重。
- 平台加载函数 `final/utils/utils.py:load_model()` 把“平台标准 checkpoint 格式”定义为：顶层包含 `state_dict`。

因此离线转换产物必须满足以下三个条件（平台代码中都有对应位置）：

1. `torch.load(dst)` 返回 `dict`，且包含键 `state_dict`。
2. `state_dict` 的 key 集合与平台 policy 的 `policy.state_dict().keys()` 完全一致。
3. 对所有 key：`state_dict[k].shape == policy.state_dict()[k].shape`，并且 `strict=True` 成功加载回同结构 policy。

本 notebook 接下来只解释两件事：

1) 为什么平台需要离线转换；
2) 离线转换脚本应该怎么写、为什么要这样写（指出平台代码中的证据位置）。


In [1]:
# --- 路径定位：保证 notebook 在 repo 根目录或任意子目录运行都能找到 final/ ---

from pathlib import Path

def find_final_dir(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for p in [here, *here.parents]:
        cand = p / "final"
        if cand.exists() and cand.is_dir():
            return cand
    raise FileNotFoundError("Cannot locate final/ directory from current working directory")

final_dir = find_final_dir()
repo_root = final_dir.parent
print("repo_root =", repo_root)
print("final_dir =", final_dir)


repo_root = /mnt/d/Study/3/neural-solver-selection
final_dir = /mnt/d/Study/3/neural-solver-selection/final


In [2]:
# --- 本 notebook 使用的“真实文件”路径 ---

author_moe = final_dir / "pretrained" / "pretrained" / "reld_moe_light" / "epoch-5000.pt"
author_mtl = final_dir / "pretrained" / "pretrained" / "reld_mtl" / "epoch-5000.pt"
my_log = final_dir / "logs" / "tsplib_70" / "lehd" / "my.txt"

for p in [author_moe, author_mtl, my_log]:
    assert p.exists(), f"Missing: {p}"

print("author_moe =", author_moe)
print("author_mtl =", author_mtl)
print("my_log     =", my_log)


author_moe = /mnt/d/Study/3/neural-solver-selection/final/pretrained/pretrained/reld_moe_light/epoch-5000.pt
author_mtl = /mnt/d/Study/3/neural-solver-selection/final/pretrained/pretrained/reld_mtl/epoch-5000.pt
my_log     = /mnt/d/Study/3/neural-solver-selection/final/logs/tsplib_70/lehd/my.txt


## 0.1 为什么要进行离线转换（平台层面的明确需求）

离线转换是平台预训练模型管理方式的一部分，证据在平台文档与入口代码中是明确的：

- `final/pretrained/README.md` 写明：预训练模型会被下载后转换成 EasyNCO 需要的格式再使用。
- `final/eval.py` 的评测流程只接受一个输入：`settings.test_loader` 给出的 checkpoint 文件路径。
- `final/utils/utils.py:load_model()` 明确支持顶层 `state_dict`（平台格式）。

离线转换带来的结果是确定的：

- 评测阶段只加载 `state_dict`，不依赖作者 checkpoint 的 `epoch/optimizer/problem/step` 等字段。
- `strict=True` 校验在转换阶段完成：key 与 shape 必须完全一致；错误在转换阶段暴露。


In [11]:
# 展示平台对预训练模型的约定（final/pretrained/README.md）

from pathlib import Path

readme = (final_dir / "pretrained" / "README.md").read_text(errors="ignore")
print(readme)


## Note

This is for storing pretrained models for benchmarking purposes, and each model is from the original repository provided by the authors.

We manually download the models and convert them to the format required by EasyNCO. 

We also provide a Google drive link for all converted models, which can be downloaded and used directly.

[Google Drive](https://drive.google.com/drive/folders/1ElpVg7-uPrJf-gqEjDev4rTvOaJ91HoX?usp=sharing)


## 0.2 平台代码中哪里规定了离线转换的目标格式

下面直接打印平台关键代码片段（带行号），对应离线转换产物的三个硬性条件：

- `final/eval.py`：评测入口如何构造 `full_model_path` 并调用加载。
- `final/utils/utils.py`：`load_model()` 如何定义并识别顶层 `state_dict`。
- `final/settings/mtreld_settings.yaml`：`test_loader` 如何指定 checkpoint 路径。


In [13]:
# 打印平台关键代码片段（带行号）

from pathlib import Path


def show(path: Path, start: int, end: int) -> None:
    lines = path.read_text(errors="ignore").splitlines()
    print(f"--- {path} ---")
    for i in range(start, end + 1):
        if 1 <= i <= len(lines):
            print(f"{i:>4d}: {lines[i-1]}")

# eval.py：实例化 policy + 调用 load_model
show(final_dir / "eval.py", 78, 110)

print("")

# utils.py：load_model 对 state_dict/model_state_dict 的分支
show(final_dir / "utils" / "utils.py", 255, 290)

print("")

# settings：test_loader 指定 checkpoint 路径（以 mtreld 为例）
settings_path = final_dir / "settings" / "mtreld_settings.yaml"
settings_lines = settings_path.read_text(errors="ignore").splitlines()
idx = next(i for i,l in enumerate(settings_lines, start=1) if l.strip().startswith("test_loader"))
show(settings_path, idx, min(idx + 12, len(settings_lines)))


--- /mnt/d/Study/3/neural-solver-selection/final/eval.py ---
  78:                                          strategy=strategy,
  79:                                          logger=logger)
  80: 
  81:         test_loader = model_settings.test_loader
  82: 
  83:         full_model_path = os.sep.join([root_dir, test_loader.model_dirpath,
  84:                                        test_loader.model_filename])
  85: 
  86:         policy = hydra.utils.instantiate(model_settings.model)
  87:         policy = load_model(policy, full_model_path,device=cuda_device,model_name=cfg.model)
  88:         # if cfg.model=="glop":
  89:         #     model_settings.initialization["params"]=cfg.settings["global_params"]
  90:         #     model_settings.iteration["params"]=cfg.settings["global_params"]
  91:         initialization = hydra.utils.instantiate(
  92:             model_settings.initialization, policy=policy
  93:         )
  94:         iteration = hydra.utils.instantiate(model_setting

## 0.3 离线转换脚本应该怎样写（不调用 `load_model`）

离线转换脚本的写法由平台评测入口决定：`eval.py` 评测时会先实例化 policy，再把权重加载进去。

因此离线转换脚本必须按以下步骤写（每一步都是必要步骤）：

1. 用与你评测相同的 settings YAML（Hydra `_target_`）实例化目标 policy，得到 `policy.state_dict()`。
2. `torch.load(src)` 读取源 checkpoint，并从中取出源参数字典（本仓库作者文件包含 `model_state_dict`）。
3. 以目标 key 驱动构造 `converted_state_dict`：遍历 `policy.state_dict()` 的每个 key，必须找到一个源 tensor 且 shape 完全一致。
4. `torch.save({"state_dict": converted_state_dict, "meta": {...}}, dst)` 生成平台标准 checkpoint。
5. 再实例化一个 fresh policy，执行 `fresh.load_state_dict(converted_state_dict, strict=True)` 作为验收。

这套写法完全不依赖在线兼容逻辑：它只依赖平台 policy 的 state_dict 定义与 PyTorch 的 strict load 行为。


In [15]:
# 离线转换模板：不调用 load_model
# 运行环境：与你运行 final/eval.py 的环境一致（需要 torch、hydra、omegaconf）。

import sys
import types
from pathlib import Path

# 让 import EasyNCO.* 强制解析到 final/（避免导入到仓库顶层 EasyNCO/ 目录）
local_pkg = types.ModuleType("EasyNCO")
local_pkg.__path__ = [str(final_dir)]
sys.modules["EasyNCO"] = local_pkg
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch
import hydra
from omegaconf import OmegaConf


def instantiate_policy_from_settings(settings_yaml: Path):
    cfg = OmegaConf.load(str(settings_yaml))
    assert "model" in cfg, f"missing `model` section in {settings_yaml}"
    return hydra.utils.instantiate(cfg["model"])


def offline_convert_author_ckpt(
    *,
    src_ckpt: Path,
    dst_ckpt: Path,
    settings_yaml: Path,
    src_state_key: str = "model_state_dict",
) -> None:
    # Convert src -> dst. Output format: {"state_dict": <target_state_dict>, "meta": {...}}

    policy = instantiate_policy_from_settings(settings_yaml)
    target_sd = policy.state_dict()

    loaded = torch.load(src_ckpt, map_location="cpu")
    assert isinstance(loaded, dict), type(loaded)
    assert src_state_key in loaded, f"missing {src_state_key!r}, top_level_keys={list(loaded.keys())}"
    src_sd = loaded[src_state_key]
    assert isinstance(src_sd, dict), type(src_sd)

    # 目标 key 驱动：每个 target key 必须找到一个同 shape 的源 tensor
    converted = {}
    missing = []
    shape_mismatch = []

    for k, v_tgt in target_sd.items():
        if k not in src_sd:
            missing.append(k)
            continue
        v_src = src_sd[k]
        if tuple(v_src.shape) != tuple(v_tgt.shape):
            shape_mismatch.append((k, tuple(v_src.shape), tuple(v_tgt.shape)))
            continue
        converted[k] = v_src.detach().cpu()

    if missing or shape_mismatch:
        print("[FAIL] cannot convert")
        print("missing keys:", len(missing))
        for k in missing[:50]:
            print("  ", k)
        print("shape mismatch:", len(shape_mismatch))
        for k, s_src, s_tgt in shape_mismatch[:50]:
            print("  ", k, "src=", s_src, "tgt=", s_tgt)

        set_src = set(src_sd.keys())
        set_tgt = set(target_sd.keys())
        only_src = sorted(set_src - set_tgt)
        only_tgt = sorted(set_tgt - set_src)
        print("only_in_src:", len(only_src))
        for k in only_src[:50]:
            print("  ", k)
        print("only_in_tgt:", len(only_tgt))
        for k in only_tgt[:50]:
            print("  ", k)

        raise RuntimeError("offline conversion failed; write explicit key mapping to resolve the diff")

    dst_ckpt.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "state_dict": converted,
        "meta": {
            "source": str(src_ckpt),
            "settings": str(settings_yaml),
            "src_state_key": src_state_key,
        },
    }
    torch.save(payload, dst_ckpt)

    # strict 回读验证：这是离线转换的验收
    fresh = instantiate_policy_from_settings(settings_yaml)
    ckpt2 = torch.load(dst_ckpt, map_location="cpu")
    res = fresh.load_state_dict(ckpt2["state_dict"], strict=True)
    assert not getattr(res, "missing_keys", []), res.missing_keys
    assert not getattr(res, "unexpected_keys", []), res.unexpected_keys

    print("[OK] converted")
    print("  src:", src_ckpt)
    print("  dst:", dst_ckpt)
    print("  keys:", len(converted))


# 示例（取消注释后运行）：
offline_convert_author_ckpt(
    src_ckpt=author_moe,
    dst_ckpt=final_dir / "pretrained" / "converted" / "mtreld_epoch-5000.pt",
    settings_yaml=final_dir / "settings" / "mtreld_settings.yaml",
)

offline_convert_author_ckpt(
    src_ckpt=author_mtl,
    dst_ckpt=final_dir / "pretrained" / "converted" / "mtreld_mtl_epoch-5000.pt",
    settings_yaml=final_dir / "settings" / "mtreld_mtl_settings.yaml",
)


[OK] converted
  src: /mnt/d/Study/3/neural-solver-selection/final/pretrained/pretrained/reld_moe_light/epoch-5000.pt
  dst: /mnt/d/Study/3/neural-solver-selection/final/pretrained/converted/mtreld_epoch-5000.pt
  keys: 164
[OK] converted
  src: /mnt/d/Study/3/neural-solver-selection/final/pretrained/pretrained/reld_mtl/epoch-5000.pt
  dst: /mnt/d/Study/3/neural-solver-selection/final/pretrained/converted/mtreld_mtl_epoch-5000.pt
  keys: 68


## 1. 转换前格式（作者 checkpoint）：`epoch-5000.pt`

### 1.1 这两个文件的“外部容器格式”是确定的

这两个文件：
- `final/pretrained/pretrained/reld_moe_light/epoch-5000.pt`
- `final/pretrained/pretrained/reld_mtl/epoch-5000.pt`

都是 **torch 的 zip 序列化容器**（`zipfile.is_zipfile(...) == True`）。
它们内部都有一个 `epoch-5000/data.pkl`，这是 torch 保存对象结构的 pickle 元数据。

下面用标准库 `zipfile` 直接检验，不依赖 torch。

In [3]:
import zipfile

def inspect_torch_zip(pt_path: Path) -> dict:
    assert zipfile.is_zipfile(pt_path), f"Not a torch zip checkpoint: {pt_path}"
    with zipfile.ZipFile(pt_path, "r") as z:
        names = z.namelist()
        data_pkl = next(n for n in names if n.endswith("data.pkl"))
        data = z.read(data_pkl)

    return {
        "zip_entries": len(names),
        "data_pkl": data_pkl,
        "data_pkl_bytes": len(data),
        "has_model_state_dict_str": (b"model_state_dict" in data),
        "has_optimizer_state_dict_str": (b"optimizer_state_dict" in data),
        "has_epoch_str": (b"epoch" in data),
        "has_problem_str": (b"problem" in data),
        "has_step_str": (b"step" in data),
        "has_encoder_prefix_str": (b"encoder." in data),
        "has_decoder_prefix_str": (b"decoder." in data),
        "has_policy_prefix_str": (b"policy." in data),
    }

info_moe = inspect_torch_zip(author_moe)
info_mtl = inspect_torch_zip(author_mtl)

print("[MOE_LIGHT]", info_moe)
print("[MTL]     ", info_mtl)


[MOE_LIGHT] {'zip_entries': 624, 'data_pkl': 'epoch-5000/data.pkl', 'data_pkl_bytes': 101379, 'has_model_state_dict_str': True, 'has_optimizer_state_dict_str': True, 'has_epoch_str': True, 'has_problem_str': True, 'has_step_str': True, 'has_encoder_prefix_str': True, 'has_decoder_prefix_str': True, 'has_policy_prefix_str': False}
[MTL]      {'zip_entries': 276, 'data_pkl': 'epoch-5000/data.pkl', 'data_pkl_bytes': 56938, 'has_model_state_dict_str': True, 'has_optimizer_state_dict_str': True, 'has_epoch_str': True, 'has_problem_str': True, 'has_step_str': True, 'has_encoder_prefix_str': True, 'has_decoder_prefix_str': True, 'has_policy_prefix_str': False}


### 1.2 以上代码在本仓库的实际输出（固定值）

对本仓库这两个文件，上面 cell 的实际结果是：

- `reld_moe_light/epoch-5000.pt`
  - `zip_entries = 624`
  - `data_pkl = "epoch-5000/data.pkl"`
  - `data_pkl_bytes = 101379`
  - `has_model_state_dict_str = True`
  - `has_optimizer_state_dict_str = True`
  - `has_epoch_str = True`, `has_problem_str = True`, `has_step_str = True`
  - `has_encoder_prefix_str = True`, `has_decoder_prefix_str = True`, `has_policy_prefix_str = False`

- `reld_mtl/epoch-5000.pt`
  - `zip_entries = 276`
  - `data_pkl = "epoch-5000/data.pkl"`
  - `data_pkl_bytes = 56938`
  - `has_model_state_dict_str = True`
  - `has_optimizer_state_dict_str = True`
  - `has_epoch_str = True`, `has_problem_str = True`, `has_step_str = True`
  - `has_encoder_prefix_str = True`, `has_decoder_prefix_str = True`, `has_policy_prefix_str = False`

这些值来自对仓库文件的直接读取（标准库 zipfile + 字节串搜索）。

### 1.3 这两个作者 checkpoint 的“参数字典长度”也是确定的

这点不依赖 torch 直接加载权重，依赖的是你实际推理日志（平台 `load_model()` 打印的确定信息）。

平台日志 `final/logs/tsplib_70/lehd/my.txt` 中：

- `mtreld` 每个任务都会打印：`model_state_dict found with 164 parameters`
- `mtreld_mtl` 每个任务都会打印：`model_state_dict found with 68 parameters`

下面用正则从日志里把它们提取出来。

In [4]:
import re
from collections import Counter

text = my_log.read_text(errors="ignore")
pat = re.compile(r"Loaded ReLD author format: model_state_dict found with (\d+) parameters")
nums = [int(m.group(1)) for m in pat.finditer(text)]
print("unique_counts:", sorted(set(nums)))
print("count_by_value:", dict(Counter(nums)))


unique_counts: [68, 164]
count_by_value: {164: 16, 68: 16}


#### 本仓库的实际结果（固定值）

- `unique_counts = [68, 164]`
- `count_by_value = {164: 16, 68: 16}`

解释：

- 16 个 VRP 任务各跑一次 `mtreld`，每次加载作者 checkpoint，都走 `model_state_dict` 分支，且 `len(model_state_dict)=164`。
- 16 个 VRP 任务各跑一次 `mtreld_mtl`，每次加载作者 checkpoint，都走 `model_state_dict` 分支，且 `len(model_state_dict)=68`。


### 1.4（可选）用 torch 直接打印作者 checkpoint 的顶层 keys

下面 cell 会做两件事：

1) `torch.load(author_ckpt)` 直接读取顶层 dict。
2) 打印 `ckpt.keys()` 和 `len(ckpt['model_state_dict'])`。

你在运行 `final/eval.py` 的环境里有 torch，因此该 cell 会直接跑通并输出顶层结构。

In [5]:
def torch_inspect_ckpt(pt_path: Path, max_show: int = 10) -> None:
    import torch

    ckpt = torch.load(pt_path, map_location="cpu")
    print("path:", pt_path)
    print("type:", type(ckpt))
    print("top_level_keys:", list(ckpt.keys()))

    msd = ckpt["model_state_dict"]
    print("len(model_state_dict)=", len(msd))
    keys = list(msd.keys())[:max_show]
    print(f"first_{max_show}_param_keys:")
    for k in keys:
        v = msd[k]
        shape = getattr(v, "shape", None)
        print("  ", k, "shape=", tuple(shape) if shape is not None else type(v))


torch_inspect_ckpt(author_moe)
print("-" * 80)
torch_inspect_ckpt(author_mtl)


path: /mnt/d/Study/3/neural-solver-selection/final/pretrained/pretrained/reld_moe_light/epoch-5000.pt
type: <class 'dict'>
top_level_keys: ['epoch', 'problem', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'result_log']
len(model_state_dict)= 164
first_10_param_keys:
   encoder.embedding_depot.weight shape= (128, 2)
   encoder.embedding_depot.bias shape= (128,)
   encoder.embedding_node.weight shape= (128, 5)
   encoder.embedding_node.bias shape= (128,)
   encoder.layers.0.Wq.weight shape= (128, 128)
   encoder.layers.0.Wk.weight shape= (128, 128)
   encoder.layers.0.Wv.weight shape= (128, 128)
   encoder.layers.0.multi_head_combine.weight shape= (128, 128)
   encoder.layers.0.multi_head_combine.bias shape= (128,)
   encoder.layers.0.feedForward.w_gate shape= (128, 4)
--------------------------------------------------------------------------------
path: /mnt/d/Study/3/neural-solver-selection/final/pretrained/pretrained/reld_mtl/epoch-5000.pt
type: <class 'dict'>
t

## 1.5 `model_state_dict`（作者）→ `state_dict`（平台）的 key 变化（逐项对齐统计）

下面这个分析 cell 对 `mtreld` 与 `mtreld_mtl` 各执行一次，输出完全由本仓库真实文件计算得到：

1. 读取作者 checkpoint（`torch.load(...)`），取出 `model_state_dict`：
   - 打印 `len(model_state_dict)`
   - 打印 key 的前缀统计（例如 `encoder.`/`decoder.`）
   - 打印 key 样例
2. 用 settings YAML 实例化平台 policy（Hydra `_target_`）：
   - 打印 `len(policy.state_dict())`
   - 打印 key 的前缀统计
   - 打印 key 样例
3. 计算集合差：
   - `only_in_author = author_keys - policy_keys`
   - `only_in_policy = policy_keys - author_keys`
   - 打印差集计数与样例
4. 按 `final/utils/utils.py:load_model()` 的匹配顺序做一次“纯分析匹配”（不改权重）：
   - 对每个 `policy_key` 找到作者 `model_state_dict` 中实际会匹配的 key（同 shape 才算匹配）
   - 输出 `target_key != matched_key` 的条目（这就是“key 发生了哪些变化”）
   - 统计每种匹配规则命中次数（`exact` / `strip_encoder_prefix` / ...）

该 cell 的匹配逻辑与 `final/utils/utils.py:load_model()` 使用的 key 规则一致。


In [10]:
# --- 1.5: keys 变化：作者 model_state_dict -> 平台 policy.state_dict（按 load_model 规则逐项对齐） ---
# 该 cell 需要 torch / hydra / omegaconf（与你运行 final/eval.py 的环境一致）。

import sys
import types
from collections import Counter
from pathlib import Path

# 让 import EasyNCO.* 强制解析到本仓库的 final/（避免导入到顶层 EasyNCO/ 目录）
local_pkg = types.ModuleType("EasyNCO")
local_pkg.__path__ = [str(final_dir)]
sys.modules["EasyNCO"] = local_pkg
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch
import hydra
from omegaconf import OmegaConf


def prefix_stats(keys: list[str], prefixes: list[str]) -> dict[str, int]:
    c: Counter[str] = Counter()
    for k in keys:
        hit = None
        for p in prefixes:
            if k.startswith(p):
                hit = p
                break
        c[hit or "<other>"] += 1
    return dict(c)


def load_author_model_state_dict(path: Path):
    ckpt = torch.load(path, map_location="cpu")
    assert isinstance(ckpt, dict), type(ckpt)
    assert "model_state_dict" in ckpt, f"top_level_keys={list(ckpt.keys())}"
    msd = ckpt["model_state_dict"]
    assert isinstance(msd, dict), type(msd)
    return ckpt, msd


def instantiate_policy(settings_yaml: Path):
    cfg = OmegaConf.load(str(settings_yaml))
    assert "model" in cfg, f"missing `model` in {settings_yaml}"
    policy = hydra.utils.instantiate(cfg["model"])
    return policy


# 下面三段函数：严格复刻 final/utils/utils.py:load_model() 的 key 匹配规则（字符串层面）

def possible_names_for(name: str) -> list[str]:
    return [
        name,
        name.replace("encoder.", ""),
        name.replace("decoder.", ""),
        "policy." + name,
        "policy." + name.replace("encoder.", ""),
        "policy." + name.replace("decoder.", ""),
        name.replace("encoder.layers.", "layers."),
        name.replace("decoder.feed_forward.", "feed_forward."),
        "policy." + name.replace("encoder.layers.", "layers."),
        "policy." + name.replace("decoder.feed_forward.", "feed_forward."),
    ]


def match_reason(target: str, cand: str) -> str:
    if cand == target:
        return "exact"
    if cand == "policy." + target:
        return "add_policy_prefix"
    if cand == target.replace("encoder.", ""):
        return "strip_encoder_prefix"
    if cand == target.replace("decoder.", ""):
        return "strip_decoder_prefix"
    if cand == "policy." + target.replace("encoder.", ""):
        return "add_policy_prefix + strip_encoder_prefix"
    if cand == "policy." + target.replace("decoder.", ""):
        return "add_policy_prefix + strip_decoder_prefix"
    if cand == target.replace("encoder.layers.", "layers."):
        return "encoder.layers -> layers"
    if cand == target.replace("decoder.feed_forward.", "feed_forward."):
        return "decoder.feed_forward -> feed_forward"
    if cand == "policy." + target.replace("encoder.layers.", "layers."):
        return "add_policy_prefix + encoder.layers -> layers"
    if cand == "policy." + target.replace("decoder.feed_forward.", "feed_forward."):
        return "add_policy_prefix + decoder.feed_forward -> feed_forward"
    return "other"


def match_target_to_ckpt_key(target_key: str, target_tensor, ckpt_dict: dict):
    # Step 1: ckpt_name = "policy." + name if present else name
    ckpt_name = ("policy." + target_key) if ("policy." + target_key in ckpt_dict) else target_key
    if ckpt_name in ckpt_dict:
        ckpt_tensor = ckpt_dict[ckpt_name]
        assert ckpt_tensor.shape == target_tensor.shape, (
            target_key,
            ckpt_name,
            tuple(target_tensor.shape),
            tuple(ckpt_tensor.shape),
        )
        return ckpt_name, ("policy_prefix_direct" if ckpt_name != target_key else "direct")

    # Step 2: 依次尝试 possible_names；存在且 shape 一致才算匹配
    for cand in possible_names_for(target_key):
        if cand in ckpt_dict:
            ckpt_tensor = ckpt_dict[cand]
            if ckpt_tensor.shape == target_tensor.shape:
                return cand, match_reason(target_key, cand)

    return None, "unmatched"


def analyze_variant(tag: str, author_ckpt_path: Path, settings_yaml: Path, model_name: str, converted_ckpt_path: Path):
    print("=" * 120)
    print("VARIANT:", tag)
    print("author_ckpt:", author_ckpt_path)
    print("settings  :", settings_yaml)
    print("model_name:", model_name)
    print("converted :", converted_ckpt_path)

    author_ckpt, msd = load_author_model_state_dict(author_ckpt_path)
    author_top_keys = list(author_ckpt.keys())
    author_keys = list(msd.keys())

    print("[Author checkpoint]")
    print("author top-level keys:", author_top_keys)
    print("author len(model_state_dict)=", len(author_keys))
    print("author prefix stats:", prefix_stats(author_keys, ["encoder.", "decoder.", "layers.", "feed_forward.", "attr_mapping."]))
    print("author first 20 keys:")
    for k in author_keys[:20]:
        print("  ", k)

    policy = instantiate_policy(settings_yaml)
    policy_sd = policy.state_dict()
    policy_keys = list(policy_sd.keys())

    print("[Platform policy]")
    print("policy class:", type(policy))
    print("policy len(state_dict)=", len(policy_keys))
    print("policy prefix stats:", prefix_stats(policy_keys, ["encoder.", "decoder.", "layers.", "feed_forward.", "attr_mapping."]))
    print("policy first 20 keys:")
    for k in policy_keys[:20]:
        print("  ", k)

    set_author = set(author_keys)
    set_policy = set(policy_keys)
    only_author = sorted(set_author - set_policy)
    only_policy = sorted(set_policy - set_author)
    inter = set_author & set_policy

    print("[Set diff]")
    print("|author|       =", len(set_author))
    print("|policy|       =", len(set_policy))
    print("|intersection| =", len(inter))
    print("|only_author|  =", len(only_author))
    print("|only_policy|  =", len(only_policy))

    print("only_author first 30:")
    for k in only_author[:30]:
        print("  ", k)

    print("only_policy first 30:")
    for k in only_policy[:30]:
        print("  ", k)

    # 逐 policy 参数做一次匹配分析：目标 key -> 作者 key（同 shape 才算）
    reasons: Counter[str] = Counter()
    changed = []
    unmatched = []

    for k in policy_keys:
        matched, reason = match_target_to_ckpt_key(k, policy_sd[k], msd)
        reasons[reason] += 1
        if matched is None:
            unmatched.append(k)
        elif matched != k:
            changed.append((k, matched, reason))

    print("[Mapping summary] (rule order == final/utils/utils.py:load_model)")
    print("reasons:", dict(reasons))
    print("changed_key_count=", len(changed))
    print("unmatched_count  =", len(unmatched))

    print("[Changed examples] (first 50)")
    for t, m, r in changed[:50]:
        print("target:", t)
        print("match :", m)
        print("reason:", r)
        print()

    print("[Converted checkpoint]")
    if converted_ckpt_path.exists():
        conv = torch.load(converted_ckpt_path, map_location="cpu")
        print("converted top-level keys:", list(conv.keys()))
        assert "state_dict" in conv, f"converted missing state_dict: {list(conv.keys())}"
        conv_keys = set(conv["state_dict"].keys())
        print("converted len(state_dict)=", len(conv_keys))
        print("converted_keys == policy_keys:", conv_keys == set_policy)
    else:
        print("converted file not found:", converted_ckpt_path)
        print("run:", f"python convert_checkpoint_offline.py --settings {settings_yaml.relative_to(final_dir)} --model-name {model_name} --src {author_ckpt_path.relative_to(final_dir)} --dst {converted_ckpt_path.relative_to(final_dir)}")


variants = [
    {
        "tag": "mtreld (author MOE_LIGHT)  |  author model_state_dict -> platform MTReLDPolicy.state_dict",
        "author": author_moe,
        "settings": final_dir / "settings" / "mtreld_settings.yaml",
        "model_name": "mtreld",
        "converted": final_dir / "pretrained" / "converted" / "mtreld_epoch-5000.pt",
    },
    {
        "tag": "mtreld_mtl (author MTL)     |  author model_state_dict -> platform MTReLDMTLPolicy.state_dict",
        "author": author_mtl,
        "settings": final_dir / "settings" / "mtreld_mtl_settings.yaml",
        "model_name": "mtreld_mtl",
        "converted": final_dir / "pretrained" / "converted" / "mtreld_mtl_epoch-5000.pt",
    },
]

for v in variants:
    analyze_variant(v["tag"], v["author"], v["settings"], v["model_name"], v["converted"])


VARIANT: mtreld (author MOE_LIGHT)  |  author model_state_dict -> platform MTReLDPolicy.state_dict
author_ckpt: /mnt/d/Study/3/neural-solver-selection/final/pretrained/pretrained/reld_moe_light/epoch-5000.pt
settings  : /mnt/d/Study/3/neural-solver-selection/final/settings/mtreld_settings.yaml
model_name: mtreld
converted : /mnt/d/Study/3/neural-solver-selection/final/pretrained/converted/mtreld_epoch-5000.pt
[Author checkpoint]
author top-level keys: ['epoch', 'problem', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'result_log']
author len(model_state_dict)= 164
author prefix stats: {'encoder.': 154, 'decoder.': 10}
author first 20 keys:
   encoder.embedding_depot.weight
   encoder.embedding_depot.bias
   encoder.embedding_node.weight
   encoder.embedding_node.bias
   encoder.layers.0.Wq.weight
   encoder.layers.0.Wk.weight
   encoder.layers.0.Wv.weight
   encoder.layers.0.multi_head_combine.weight
   encoder.layers.0.multi_head_combine.bias
   encoder.layers.0.

/mnt/d/Study/3/neural-solver-selection/final/neural_solvers/methods/drl_hgnn/utils.py:21: SyntaxWarning: "is" with a literal. Did you mean "=="?
  elif line is "\n":


[Platform policy]
policy class: <class 'EasyNCO.neural_solvers.methods.mtreld.policy.MTReLDPolicy'>
policy len(state_dict)= 164
policy prefix stats: {'encoder.': 154, 'decoder.': 10}
policy first 20 keys:
   encoder.embedding_depot.weight
   encoder.embedding_depot.bias
   encoder.embedding_node.weight
   encoder.embedding_node.bias
   encoder.layers.0.Wq.weight
   encoder.layers.0.Wk.weight
   encoder.layers.0.Wv.weight
   encoder.layers.0.multi_head_combine.weight
   encoder.layers.0.multi_head_combine.bias
   encoder.layers.0.feedForward.w_gate
   encoder.layers.0.feedForward.w_noise
   encoder.layers.0.feedForward.mean
   encoder.layers.0.feedForward.std
   encoder.layers.0.feedForward.experts.0.fc1.weight
   encoder.layers.0.feedForward.experts.0.fc1.bias
   encoder.layers.0.feedForward.experts.0.fc2.weight
   encoder.layers.0.feedForward.experts.0.fc2.bias
   encoder.layers.0.feedForward.experts.1.fc1.weight
   encoder.layers.0.feedForward.experts.1.fc1.bias
   encoder.layers.0.f

## 2. 转换后格式（平台 checkpoint）：`{"state_dict": ...}`

离线转换脚本 `final/convert_checkpoint_offline.py` 生成的新文件顶层结构是**固定**的：

```python
{
  "state_dict": {  # 平台 policy 的 state_dict（key 与 policy.state_dict() 完全一致）
      "encoder....": tensor(...),
      "decoder....": tensor(...),
      ...
  },
  "meta": {
      "source": "<原作者ckpt路径>",
      "settings": "settings/mtreld_settings.yaml" 或 "settings/mtreld_mtl_settings.yaml",
      "model_name": "mtreld" 或 "mtreld_mtl",
  }
}
```

这里的核心变化是：

- 作者文件用 `model_state_dict` 装参数（并带 epoch/optimizer 等字段）。
- 平台文件用 `state_dict` 装参数（key 与平台实现一致）。

这使得平台推理可以直接走 `final/utils/utils.py:283` 的 `state_dict` 分支。

## 3. 在线加载（`load_model`）实际做了什么

`final/utils/utils.py:load_model()` 的实际逻辑是：

1) `torch.load(path)` 读取 checkpoint 到 `loaded_data`。
2) 按优先级选参数字典 `ckpt_dict`：
   - 若 `"model_state_dict" in loaded_data`：用 `loaded_data["model_state_dict"]`
   - elif `"state_dict" in loaded_data`：用 `loaded_data["state_dict"]`
   - else：把 `loaded_data` 当成“直接 state_dict”。
3) 遍历 `policy.state_dict()` 的每个参数 `name, param`：
   - 先查 `name` 是否在 `ckpt_dict`；在就 copy。
   - 不在就按 `possible_names`（固定列表）逐个尝试替代 key，找到且 shape 一致就 copy。
4) 如果任何一个参数最终都找不到可匹配 key，就直接报错退出（不是 silent）。

下面 cell 会把 `load_model` 的核心片段打印出来（带行号），你可以和 `utils.py` 对照阅读。

In [6]:
# 打印 load_model 的核心片段（不依赖 torch）

utils_path = final_dir / "utils" / "utils.py"
lines = utils_path.read_text(errors="ignore").splitlines()

start, end = 255, 362  # 与 `nl -ba final/utils/utils.py` 对齐的行号范围
for i in range(start, end + 1):
    if 1 <= i <= len(lines):
        print(f"{i:>4d}: {lines[i-1]}")


 255: def load_model(model, path, device, model_name):
 256:     class Placeholder:
 257:         def __init__(self, *args, **kwargs):
 258:             pass
 259:         def __getattr__(self, name):
 260:             return lambda *args, **kwargs: None
 261: 
 262:     class CustomUnpickler(pickle.Unpickler):
 263:         def find_class(self, module, name):
 264:             if module.startswith('EasyNCO'):
 265:                 return Placeholder
 266:             return super().find_class(module, name)
 267: 
 268:     custom_pickle_module = types.ModuleType('custom_pickle_module')
 269:     custom_pickle_module.Unpickler = CustomUnpickler
 270: 
 271:     load_device = "cpu" if device == 1 else f"cuda:{device[0]}"
 272: 
 273:     if model_name != 'nlns':
 274:         try:
 275:             # 🔥 ReLD移植：尝试不同的权重文件结构
 276:             loaded_data = torch.load(path, pickle_module=custom_pickle_module, map_location=load_device)
 277:             
 278:             # 检查权重文件的结构
 279:   

## 4. 离线转换（`convert_checkpoint_offline.py`）具体做了什么

离线转换脚本把“在线加载的结果”变成一个可复用的标准文件。它的步骤是确定的：

1) 用 settings YAML 实例化 policy（确保结构与推理一致）。
2) 调用现有 `load_model(policy, author_ckpt, device=1, model_name=...)` 在 CPU 上完成加载。
3) 保存 `policy.state_dict()` 为 `{"state_dict": ...}`。
4) 再实例化一个 fresh policy，`strict=True` 回读验证，保证新文件 key/shape 完全匹配。

下面 cell 打印离线脚本核心片段（带行号）。

In [7]:
conv_path = final_dir / "convert_checkpoint_offline.py"
conv_lines = conv_path.read_text(errors="ignore").splitlines()
for i, line in enumerate(conv_lines, start=1):
    if 104 <= i <= 143:
        print(f"{i:>4d}: {line}")


 104:     # Heavy imports after path setup
 105:     import torch
 106:     import hydra
 107:     from omegaconf import OmegaConf
 108: 
 109:     from EasyNCO.utils.utils import load_model
 110: 
 111:     if model_name == "nlns":
 112:         raise ValueError("This script currently targets single-policy checkpoints (not NLNS multi-file checkpoints).")
 113: 
 114:     settings_cfg = OmegaConf.load(str(settings_path))
 115:     if "model" not in settings_cfg:
 116:         raise ValueError(f"Invalid settings file (missing `model` section): {settings_path}")
 117: 
 118:     policy = hydra.utils.instantiate(settings_cfg["model"])
 119:     policy = load_model(policy, str(src_path), device=1, model_name=model_name)  # CPU
 120: 
 121:     # Save only tensors + plain metadata (avoid pickling code objects).
 122:     state_dict = {k: v.detach().cpu() for k, v in policy.state_dict().items()}
 123:     payload = {
 124:         "state_dict": state_dict,
 125:         "meta": {
 126:      

## 5. 运行离线转换（生成平台标准 checkpoint）

离线转换命令（在 `final/` 目录执行）：

### 5.1 转 `mtreld`（MOE_LIGHT）

```bash
python convert_checkpoint_offline.py \
  --settings settings/mtreld_settings.yaml \
  --model-name mtreld \
  --src pretrained/pretrained/reld_moe_light/epoch-5000.pt \
  --dst pretrained/converted/mtreld_epoch-5000.pt
```

### 5.2 转 `mtreld_mtl`（MTL）

```bash
python convert_checkpoint_offline.py \
  --settings settings/mtreld_mtl_settings.yaml \
  --model-name mtreld_mtl \
  --src pretrained/pretrained/reld_mtl/epoch-5000.pt \
  --dst pretrained/converted/mtreld_mtl_epoch-5000.pt
```

脚本成功会打印：

- `[OK] Converted: <src>`
- `-> <dst>`
- `keys: <N>`（这里的 N 等于 `len(policy.state_dict())`）


## 6. 验证：转换后文件的顶层 keys 与加载分支

离线产物的顶层 keys 是确定的：`state_dict` 与 `meta`。

当你把 `settings.test_loader.model_dirpath/model_filename` 指到离线产物后：

- `load_model()` 会走 `elif 'state_dict' in loaded_data:` 分支（`final/utils/utils.py:283`）。
- 日志会从 `🔥 Loaded ReLD author format: model_state_dict ...` 变成 `📦 Loaded platform format: state_dict ...`。

下面提供一个验证 cell（需要 torch）：

In [ ]:
from pathlib import Path

converted_moe = final_dir / "pretrained" / "converted" / "mtreld_epoch-5000.pt"
converted_mtl = final_dir / "pretrained" / "converted" / "mtreld_mtl_epoch-5000.pt"

def inspect_converted(pt_path: Path, max_show: int = 10) -> None:
    import torch
    ckpt = torch.load(pt_path, map_location="cpu")
    print("path:", pt_path)
    print("top_level_keys:", list(ckpt.keys()))
    sd = ckpt["state_dict"]
    print("len(state_dict)=", len(sd))
    for k in list(sd.keys())[:max_show]:
        v = sd[k]
        print("  ", k, "shape=", tuple(v.shape))
    print("meta:", ckpt.get("meta"))


if converted_moe.exists():
    inspect_converted(converted_moe)
else:
    print("missing:", converted_moe)
    print(
        "run:",
        "python convert_checkpoint_offline.py --settings settings/mtreld_settings.yaml --model-name mtreld --src pretrained/pretrained/reld_moe_light/epoch-5000.pt --dst pretrained/converted/mtreld_epoch-5000.pt",
    )

print("-" * 80)

if converted_mtl.exists():
    inspect_converted(converted_mtl)
else:
    print("missing:", converted_mtl)
    print(
        "run:",
        "python convert_checkpoint_offline.py --settings settings/mtreld_mtl_settings.yaml --model-name mtreld_mtl --src pretrained/pretrained/reld_mtl/epoch-5000.pt --dst pretrained/converted/mtreld_mtl_epoch-5000.pt",
    )


## 7. 为什么要这样转换（针对你这个仓库的“具体原因”）

### 7.1 你当前推理确实在加载作者 `model_state_dict`

证据来自 `final/logs/tsplib_70/lehd/my.txt`：

- `164` 和 `68` 两个参数数量各出现 `16` 次（16 个任务）。
- 这对应 `load_model()` 的 `if 'model_state_dict' in loaded_data:` 分支。

也就是说，你现在每次推理都在做一次“作者格式解析 + 参数 key 匹配 + copy”。

### 7.2 离线转换把“每次推理都要做的加载适配”变成“一次性生成标准资产”

离线转换后：

- 你把作者 ckpt 转成平台标准 `state_dict` 文件，并保存到 `final/pretrained/converted/`。
- 后续推理只加载该 `state_dict`，不再依赖作者文件的 `model_state_dict` 外壳字段。
- 离线脚本内置 `strict=True` 回读校验：转换产物与平台 policy 的 key/shape 完全一致。

这就是“离线转换”的定义：把格式差异与命名差异在推理前消掉，让推理阶段只做一件事：严格加载平台 state_dict。

## 8. 如何在推理中使用离线产物（只改配置/命令，不改代码）

两种方式：

### 8.1 直接改 settings 文件（永久生效）

- `final/settings/mtreld_settings.yaml`：把 `test_loader` 指到 `pretrained/converted/mtreld_epoch-5000.pt`
- `final/settings/mtreld_mtl_settings.yaml`：把 `test_loader` 指到 `pretrained/converted/mtreld_mtl_epoch-5000.pt`

### 8.2 在命令行覆盖（不改 YAML）

```bash
python eval.py \
  settings=mtreld_settings mode=test model=mtreld problem=cvrp scale=100 cuda=[0] \
  test_data_path=mt/CVRP/cvrp100_uniform.pkl \
  settings.test_loader.model_dirpath=pretrained/converted \
  settings.test_loader.model_filename=mtreld_epoch-5000.pt
```

当你使用离线产物后，`load_model()` 会打印：

- `📦 Loaded platform format: state_dict found with ... parameters`

这就是你判断“已切换到离线产物”的直接观测点。